# Fundamentals of Accelerator Physics and Technology
### Simulations and Measurements Lab
# Computer Lab: Simulated beam transport with quadrupole focusing — Xsuite version
##### Original authors: K. Ruisard, N. Evans and V. S. Morozov  
##### Local Xsuite remake

This notebook replaces the original Sirepo/Elegant workflow with local Python and Xsuite. Questions for students are still numbered.

The front-facing code cells expose the physics knobs that students should be able to change. Plotting, dense line slicing, matching optimization, and particle-distribution generation are kept in the helper module `uspas_labs.quadrupole_focusing`, installed from the course GitHub repo.

<style>
.answer {
    color: #b00020;
    border-left: 4px solid #b00020;
    padding-left: 0.8em;
    margin: 0.8em 0;
}
.answer table, .answer th, .answer td {
    color: #b00020;
}
.instructor-note {
    color: #7a3b00;
    border-left: 4px solid #7a3b00;
    padding-left: 0.8em;
    margin: 0.8em 0;
}
.note {
    background-color: #f6f8fa;
    padding: 0.75em;
    border-left: 4px solid #999;
}

.widget-slider, .jupyter-widgets.widget-slider {
    width: 900px;
    max-width: 100%;
}
.widget-slider .noUi-horizontal, .jupyter-widgets.widget-slider .noUi-horizontal {
    height: 12px;
}
.widget-slider .noUi-horizontal .noUi-handle,
.jupyter-widgets.widget-slider .noUi-horizontal .noUi-handle {
    width: 24px;
    height: 24px;
    right: -12px;
    top: -7px;
    border-radius: 50%;
}
.widget-slider .widget-readout, .jupyter-widgets.widget-slider .widget-readout {
    min-width: 70px;
}
</style>

## 1. Setup

At injection, or at the start of a simulation, there is an optimal spot size and angular divergence for the beam. This is the **matched condition**. In a periodic focusing structure, such as a FODO line, the matched Twiss solution repeats with the lattice period.

We describe the transverse envelope with Twiss parameters $\beta_x$, $\beta_y$, $\alpha_x$, and $\alpha_y$. The RMS beam size is related to beta and geometric emittance by

$$
\sigma_x = \sqrt{\beta_x\epsilon_x}, \qquad \sigma_y = \sqrt{\beta_y\epsilon_y}.
$$

Throughout this lab, the length notation is

$$
L\equiv \text{QF-center to QD-center distance}=2.50\ \mathrm{m},
$$

so $L$ is one **FODO half-cell** and the full QF-center to QF-center FODO period is

$$
2L=5.00\ \mathrm{m}.
$$

| Parameter | Value |
|---|---:|
| Species | Electron |
| Energy | 1 GeV |
| $\epsilon_x$ | $6\ \mathrm{mm\,mrad}$ |
| $\epsilon_y$ | $6\ \mathrm{mm\,mrad}$ |
| Half-cell length $L$ (QF center to QD center) | $2.50\ \mathrm{m}$ |
| Full FODO period $2L$ (QF center to QF center) | $5.00\ \mathrm{m}$ |
| Quadrupole geometric strength | $K_1 = 0.6\ \mathrm{m}^{-2}$ |
| Quadrupole length $L_q$ | $0.50\ \mathrm{m}$ |

The default full FODO cell starts and ends at the **center of a focusing quadrupole**. Both physical quadrupoles are represented with half-length elements so that their centers are explicit Twiss sampling boundaries:

`half QF — drift — half QD — half QD — drift — half QF`.

In one full cell, the QF centers are at $s=0$ and $s=2L=5.00\ \mathrm{m}$, while the two QD halves meet at $s=L=2.50\ \mathrm{m}$. The surface-to-surface drift between adjacent quadrupoles is $L-L_q=2.00\ \mathrm{m}$.

In this notebook, QF focuses particle slopes in $x$ and defocuses them in $y$; QD exchanges those plane roles. These labels describe the **focusing action**, not where the beam must be smallest. A quadrupole center can be a beta-function maximum or minimum because the envelope may begin converging or diverging after that symmetry point.

In [ ]:
from pathlib import Path
import importlib
import sys

HELPER_VERSION = "main"  # Replace with a release tag, e.g. "v2026-lab1", for student releases.
HELPER_REPO = "git+https://github.com/r-hensley/uspas-2026-jupyter.git"

LOCAL_DEV = Path("uspas_labs").exists()

if LOCAL_DEV:
    print("Using local development version of uspas_labs!")
    sys.path.insert(0, str(Path.cwd()))
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
else:
    # Colab/student path only
    print(f"Installing uspas_labs from {HELPER_VERSION}")
    %pip install -q --upgrade xsuite
    %pip install -q --upgrade --no-cache-dir "{HELPER_REPO}@{HELPER_VERSION}"

import numpy as np
import pandas as pd
import xtrack as xt
from IPython.display import display

from uspas_labs import quadrupole_focusing as qfh

if LOCAL_DEV:
    qfh = importlib.reload(qfh)
    print(f"Loaded quadrupole helper from: {Path(qfh.__file__).resolve()}")

try:
    from google.colab import output as colab_output
except ImportError:
    qfh.set_plotly_renderer("plotly_mimetype")
else:
    colab_output.enable_custom_widget_manager()
    qfh.set_plotly_renderer("colab")

qfh.print_versions()


## 2. Beamline matching

### A) Start with an unmatched beam

Initially, propagate a deliberately unmatched beam through a FODO cell. Use

$$
\beta_x=\beta_y=4\ \mathrm{m}, \qquad \alpha_x=\alpha_y=0.
$$

The cell below builds the FODO line using the modern Xsuite `Environment` pattern: define named elements with `env.new(...)`, assemble a line with `env.new_line(...)`, assign a reference particle, and call `line.twiss(...)`.

The output first shows optics over one full period $2L$ and the raw entrance/exit Twiss values. It then repeats the same physical cell three times so that the consequence of the mismatch is easier to see.

**Checkpoint A (ungraded).** Which of $\beta_x$, $\alpha_x$, $\beta_y$, and $\alpha_y$ fail to return to their entrance values after one full period $2L$? If this cell is repeated, what envelope behavior should that failure produce? Explain why a transport solution can be mathematically valid without being matched.

In [ ]:
# Lab length convention: L is one QF-to-QD half-cell; 2L is one full FODO period.
L = float(qfh.FODO_HALF_CELL_LENGTH)  # 2.50 m
L_q = float(qfh.FODO_QUAD_LENGTH)    # 0.50 m
fodo_period_m = 2 * L                # 5.00 m
interquad_drift_m = L - L_q          # 2.00 m surface-to-surface drift

assert np.isclose(qfh.FODO_CELL_LENGTH, 2 * L)

env = xt.Environment()
env["k1_fodo"] = 0.6
env["L_half_cell"] = L
env["L_q"] = L_q
env["half_quad_length"] = L_q / 2
env["interquad_drift_length"] = interquad_drift_m

env.new("QFh", xt.Quadrupole, length="half_quad_length", k1="k1_fodo")
env.new("D", xt.Drift, length="interquad_drift_length")
env.new("QDh", xt.Quadrupole, length="half_quad_length", k1="-k1_fodo")

fodo_cell = env.new_line(
    name="fodo_cell",
    components=["QFh", "D", "QDh", "QDh", "D", "QFh"],
)
fodo_cell.particle_ref = xt.Particles(
    p0c=1e9,
    mass0=xt.ELECTRON_MASS_EV,
    q0=-1,
)
fodo_cell.build_tracker()

assert np.isclose(fodo_cell.get_length(), 2 * L)
display(pd.DataFrame({
    "length definition": ["L: QF center to QD center", "2L: QF center to QF center"],
    "value [m]": [L, 2 * L],
}))
display(fodo_cell.get_table().to_pandas()[["name", "s_start", "s_end", "element_type"]])

unmatched_initial = dict(
    betx=4.0, alfx=0.0,
    bety=4.0, alfy=0.0,
)
tw_unmatched_cell = fodo_cell.twiss(method="4d", **unmatched_initial)
tw_unmatched_dense = qfh.twiss_dense(fodo_cell, **unmatched_initial)

qfh.plot_beta_and_sigma(
    tw_unmatched_dense,
    "Unmatched Twiss functions and RMS sizes through one full FODO period 2L",
)

unmatched_endpoint_check = pd.DataFrame({
    "Twiss quantity": ["beta_x [m]", "alpha_x", "beta_y [m]", "alpha_y"],
    "entrance": [
        tw_unmatched_cell.betx[0], tw_unmatched_cell.alfx[0],
        tw_unmatched_cell.bety[0], tw_unmatched_cell.alfy[0],
    ],
    "after one full period 2L": [
        tw_unmatched_cell.betx[-1], tw_unmatched_cell.alfx[-1],
        tw_unmatched_cell.bety[-1], tw_unmatched_cell.alfy[-1],
    ],
})
display(unmatched_endpoint_check)

fodo_unmatched_3cells = qfh.make_fodo_line(
    n_cells=3,
    k1=float(env["k1_fodo"]),
    half_cell_length=L,
)
tw_unmatched_3cells_dense = qfh.twiss_dense(
    fodo_unmatched_3cells,
    points_per_meter=30,
    **unmatched_initial,
)
qfh.plot_beta_and_sigma(
    tw_unmatched_3cells_dense,
    "Unmatched envelope through three repeated FODO periods of length 2L",
)

### B) Solve for the matched solution

Now ask Xsuite for the periodic Twiss solution of one full cell of length $2L$. For transverse coordinate $u=x$ or $y$,

$$
u(s)=\sqrt{\epsilon_u\beta_u(s)}
\cos\!\left[\psi_u(s)+\phi_{u,0}\right].
$$

Here $\beta_u(s)$ controls the local amplitude and $\psi_u(s)$ is accumulated betatron phase. The phase advance through one full period $2L$ is

$$
\mu_{u,2L}
=\Delta\psi_u(2L),
\qquad
q_{u,2L}
=\frac{\mu_{u,2L}}{2\pi}.
$$

Thus $\mu_{u,2L}$ is measured in radians per full period $2L$, while $q_{u,2L}$ is the number of oscillations per full period $2L$. One complete oscillation corresponds to $2\pi$ radians.

For the line spanning one full period $2L$, Xsuite reports

$$
\texttt{qx}=q_{x,2L},
\qquad
\texttt{qy}=q_{y,2L}.
$$

**Self-check:** If $\beta(s)=10\ \mathrm{m}$ everywhere along a $100\ \mathrm{m}$ beamline, use $\Delta\psi=\int ds/\beta(s)$ to find the total phase advance and number of oscillations.

**Q0) Convert the reported $q_x$ and $q_y$ into phase advance per full period $2L$ in radians and degrees. Why are the horizontal and vertical results equal for this ideal FODO cell? Describe in your own words the difference between "phase advance per cell" and "tune".**

<details class="note">
<summary><strong>Theory note: normalized phase space, matrices, and ring tune</strong></summary>

In normalized phase-space coordinates, $\psi_u$ is the angle through which the particle has rotated. The geometric angle in the unnormalized $(u,u')$ plane is not generally equal to $\psi_u$, because the ellipse can be stretched and tilted by $\beta_u$ and $\alpha_u$.

The Twiss integral and exact linear transfer matrix over $2L$ give the same phase advance:

$$
\mu_{u,2L}
=\int_0^{2L}\frac{ds}{\beta_u(s)},
\qquad
\cos\mu_{u,2L}
=\frac{\operatorname{Tr}(M_{u,2L})}{2}.
$$

For a ring containing $N$ identical full periods of length $2L$, the full-ring tune is

$$
Q_u=\nu_u=Nq_{u,2L}.
$$

The notation is not universal, so always identify whether a reported phase or tune refers to one full period $2L$, a finite beamline, or one complete revolution.

</details>

In [ ]:
tw_cell = fodo_cell.twiss(method="4d")
tw_cell_dense = qfh.twiss_dense(fodo_cell)

qfh.plot_beta_and_sigma(tw_cell_dense, "Matched Twiss functions and RMS sizes for one full FODO period 2L")
pd.DataFrame({
    "Xsuite output": ["qx", "qy"],
    "oscillations per full period 2L": [float(tw_cell.qx), float(tw_cell.qy)],
})

### Try it: change the FODO strength

Move the sliders and watch how the beta functions, beam size, and phase advance change.


In [ ]:
qfh.interactive_fodo_strength()


The thin-lens approximation gives the following FODO estimates when $L$ is the half-cell length and $2L$ is the full period:

$$
\beta_{\max}=2L \frac{1 + \sin(\mu/2)}{\sin \mu},
\qquad
\beta_{\min}=2L \frac{1 - \sin(\mu/2)}{\sin \mu}.
$$

In this notebook there is only one FODO spacing symbol:

$$
L=2.50\ \mathrm{m}\quad\text{(QF center to QD center)},
\qquad
2L=5.00\ \mathrm{m}\quad\text{(full FODO period)}.
$$

For a quadrupole of length $L_q$ and geometric strength $k_1$, the thin-lens focal-length magnitude is

$$
f=\frac{1}{|k_1|L_q}.
$$

The symmetric thin-lens FODO phase relation is

$$
\sin\left(\frac{\mu_{\mathrm{thin}}}{2}\right)=\frac{L}{2f},
\qquad
q_{\mathrm{thin}}=\frac{\mu_{\mathrm{thin}}}{2\pi}.
$$

In Q1, use the **Xsuite phase advance** inside the thin-lens beta formulas. This is intentionally a controlled, same-phase-advance comparison rather than a fully independent thin-lens prediction: it isolates the beta-shape error caused by replacing distributed focusing with instantaneous kicks. Q6 later makes the independent thin-lens phase prediction from $k_1$, $L_q$, and the half-cell length $L$.

**Q1) Determine $\beta_{\min}$ and $\beta_{\max}$ in two ways, then compare them.**

- Calculate the thin-lens values from your Q0 phase advance using the full-period prefactor $2L$.
- Obtain the thick-lens Xsuite extrema from the matched beta plot or Twiss arrays.
- For each extremum, calculate $(\beta_{\mathrm{thin}}-\beta_{\mathrm{Xsuite}})/\beta_{\mathrm{Xsuite}}$ as a percentage and state whether the thin-lens model overestimates or underestimates it.

In [ ]:
# Calculation workspace for Q1. L was defined above as the half-cell length.
half_cell_length_m = L
fodo_period_m = 2 * L
mu_cell_rad = 2 * np.pi * float(tw_cell.qx)
beta_x_dense = np.asarray(tw_cell_dense.betx, dtype=float)
beta_y_dense = np.asarray(tw_cell_dense.bety, dtype=float)

# Add your thin-lens and Xsuite-extrema calculations below.

### **Q2) Test the effect of finite quadrupole length**

Before running either display, predict whether increasing the physical quadrupole length—while holding the half-cell length $L$, the full period $2L$, and the phase advance fixed—will improve or worsen the thin-lens approximation.

The scan and slider both adjust $k_1$ to preserve the default phase advance over one full period $2L$ while $L_q$ changes. Use them to answer descriptively in your own words:

- As $L_q$ increases, does $\beta(s)$ change more or less throughout the magnet?
- How does the discrepancy in $\beta_{\max}$ change?
- Cite one short- and one long-quadrupole case, then explain the trend in terms of localized versus distributed focusing.

In [ ]:
quad_length_scan = qfh.quad_length_fixed_phase_table(target_q=tw_cell.qx)
qfh.plot_quad_length_scan(quad_length_scan, target_q=tw_cell.qx)


In [ ]:
qfh.interactive_quad_length_effect(target_q=tw_cell.qx)


**Q3) Convert the matched beta functions into an RMS beam-size summary.**

Use $\sigma=\sqrt{\beta\epsilon}$ with $\epsilon_x=\epsilon_y=6\ \mathrm{mm\,mrad}$ to calculate:

- the minimum and maximum $\sigma_x$ and $\sigma_y$;
- the maximum aspect ratio $\sigma_x/\sigma_y$.

Explain why the horizontal and vertical minimum and maximum values are equal even though the extrema occur at different positions.

In [ ]:
matched_sizes = qfh.add_beam_sizes(tw_cell_dense)
# The first rows show the available coordinates and units. Use the full arrays
# to calculate the minima, maxima, and maximum aspect ratio.
matched_sizes[["s", "sigma_x_mm", "sigma_y_mm"]].head()


**Q4) Use the RMS-size curves and lattice strip to determine where the beam is round and where each plane is largest and smallest. Give each $s$ first in terms of the half-cell length $L$ and then in meters, along with the corresponding element or region. Explain why the horizontal and vertical extrema exchange roles between QF and QD.**

Identify crossings and extrema from the evidence in the plot rather than from an automated location summary.

In [ ]:
qfh.plot_sigmas(tw_cell_dense, "Matched RMS sizes: use crossings and extrema to answer Q4")


### C) Matched beam propagation down a 100 m FODO beamline

Now repeat the same full FODO period 20 times. Since each cell has length $2L=5.00\ \mathrm{m}$,

$$
20(2L)=20(5.00\ \mathrm{m})=100\ \mathrm{m}.
$$

**Q5) Use the 100 m RMS-envelope plot to verify that the matched second moments repeat every full period $2L=5.00\ \mathrm{m}$. Does that periodic envelope imply that an individual particle returns to the same $(x,x')$ after every cell? Explain the distinction between a periodic matched envelope and betatron phase advance.**

In [ ]:
beamline_length_m = 100.0
n_fodo_cells = int(round(beamline_length_m / (2 * L)))
assert np.isclose(n_fodo_cells * (2 * L), beamline_length_m)

fodo_100m = qfh.make_fodo_line(
    n_cells=n_fodo_cells,
    k1=0.6,
    half_cell_length=L,
)
tw_100m_dense = qfh.twiss_dense(fodo_100m, points_per_meter=10)

qfh.plot_sigmas(
    tw_100m_dense,
    f"Matched RMS beam sizes through {n_fodo_cells} FODO periods, each of length 2L",
)

**Q6) Determine the accumulated horizontal oscillations and phase advance per full period $2L$ in two ways.**

The 100 m line contains

$$
N_{\mathrm{cells}}
=\frac{100\ \mathrm{m}}{2L}
=\frac{100\ \mathrm{m}}{5.00\ \mathrm{m}}
=20
$$

full FODO periods. Here $q_{x,2L}$ means oscillations per full period $2L$, while $\nu_{x,\mathrm{line}}$ means the oscillations accumulated across this finite beamline. Neither is a full-ring tune unless the line represents one complete revolution.

**Q6a — Method 1 — observed oscillations**

- Estimate the number of complete plus partial oscillations directly from $x(s)$ and the phase-space rotation. Report the complete cycles and remaining fraction that you counted; do **not** obtain Method 1 by multiplying the earlier Xsuite $q_x$.
- Convert the observation into oscillations per full period using

$$
q_{x,\mathrm{observed}}=\frac{\nu_{x,\mathrm{line}}}{N_{\mathrm{cells}}},
\qquad
\mu_{x,\mathrm{observed}}=2\pi q_{x,\mathrm{observed}}.
$$

**Q6b — Method 2 — thin-lens prediction**

- Calculate $f=1/(|k_1|L_q)$ using $|k_1|=0.60\ \mathrm{m}^{-2}$ and $L_q=0.50\ \mathrm{m}$.
- With the half-cell length $L=2.50\ \mathrm{m}$, calculate

$$
\mu_{\mathrm{thin}}=2\arcsin\left(\frac{L}{2f}\right),
\qquad
q_{\mathrm{thin}}=\frac{\mu_{\mathrm{thin}}}{2\pi},
\qquad
\nu_{\mathrm{thin,line}}=20q_{\mathrm{thin}}.
$$

Compare the observed result with both the independent thin-lens prediction and the earlier Xsuite value for one full period $2L$. Calculate the thin-lens-versus-observed percent difference and explain why the finite-length Xsuite lattice differs from the instantaneous-kick approximation.

> A bare $\arccos(x_{\mathrm{final}}/x_0)$ is not a safe cycle counter: it loses the phase-space quadrant and the number of completed rotations. Count complete cycles plus the remaining fraction, or use an unwrapped angle formed from normalized phase-space coordinates.

In [ ]:
tw_centroid = qfh.centroid_orbit(fodo_100m, x0=1e-3, px0=0.0)
qfh.plot_centroid(tw_centroid)


### D) Propagation of a mismatched beam

Initialize the beam with a 10% beta mismatch relative to the matched solution:

$$
\beta_x = 1.1\beta_{x,\mathrm{matched}},\qquad
\beta_y = 1.1\beta_{y,\mathrm{matched}},
$$

while keeping the matched alpha values.

The **smooth approximation** replaces the rapidly alternating FODO focusing with an equivalent average focusing strength. An individual particle coordinate varies like $x\propto\cos\psi$, while the envelope is determined by second moments such as $\langle x^2\rangle$. Because

$$
\cos^2\psi=\frac{1}{2}\left(1+\cos2\psi\right),
$$

a mismatch in the envelope oscillates at approximately twice the single-particle betatron phase rate. If the particle motion accumulates $\nu_{x,\mathrm{line}}$ oscillations over the beamline, then

$$
\nu_{\mathrm{envelope}}\simeq2\nu_{x,\mathrm{line}}.
$$

The real FODO envelope also contains rapid cell-by-cell variation. Count the slower mismatch modulation at equivalent positions in successive cells, rather than every local maximum inside a cell.

**Q7) Before running the plot, use Q6 to predict the number of slow envelope-mismatch oscillations over 100 m. Then count them from the boundary-sampled envelope. Does the observation agree with $\nu_{\mathrm{envelope}}\simeq2\nu_{x,\mathrm{line}}$?**

In [ ]:
tw_mismatch_dense = qfh.twiss_dense(
    fodo_100m,
    points_per_meter=10,
    betx=float(tw_cell.betx[0]) * 1.10,
    alfx=float(tw_cell.alfx[0]),
    bety=float(tw_cell.bety[0]) * 1.10,
    alfy=float(tw_cell.alfy[0]),
)

qfh.plot_mismatch(tw_100m_dense, tw_mismatch_dense, "Matched and 10% mismatched RMS beam envelopes")
qfh.plot_cell_boundary_envelope(tw_mismatch_dense)


### Try it: change the injection mismatch

The slider changes the initial beta mismatch factor. Values closer to 1 are closer to matched; values farther from 1 produce stronger beating.


In [ ]:
qfh.interactive_mismatch(tw_cell)
# this one can take a while to load after changing the slider

### E) Transition between different FODO sections

Build a hybrid beamline containing 10 full cells with $|k_1|=0.6\ \mathrm{m}^{-2}$ followed by 10 full cells with $|k_1|=0.5\ \mathrm{m}^{-2}$. Start with the periodic Twiss parameters of the stronger cell. The lattice changes after ten periods of length $2L$:

$$
s_{\mathrm{transition}}=10(2L)=20L=50\ \mathrm{m}.
$$

The first plot shows the detailed $\beta_x(s)$ and $\beta_y(s)$ functions. The second samples only equivalent QF-center boundaries, separated by $2L$, and displays

$$
100\left(\frac{\beta}{\beta_{\mathrm{matched,local}}}-1\right)
$$

in both planes. Sampling the same optical location in every cell removes the ordinary within-cell modulation: zero means locally matched, while oscillation away from zero is beta beating. Emittance is conserved because ideal linear symplectic transport preserves it.

**Q8a)**
**Use the plots to describe what changes at $s=10(2L)=20L=50\ \mathrm{m}$. Cite at least one specific boundary-sampled deviation or peak/trough after the transition. Explain why the beam is matched in the first section but not the second. (What does "matched" mean in this case?)**

**Q8b)**
**Can emittance be conserved throughout this lattice even if both $\beta_x$ and $\beta_y$ increase at the same time? Describe in your own words why or why not. Are we talking about 2D phase space being conserved, or full 6D phase space?**

In [ ]:
hybrid_cells_per_section = 10
hybrid_transition_s_m = hybrid_cells_per_section * (2 * L)

hybrid_line = qfh.make_hybrid_fodo_line(
    first_cells=hybrid_cells_per_section,
    second_cells=hybrid_cells_per_section,
    k1_first=0.6,
    k1_second=0.5,
)

tw_hybrid_from_strong_match = qfh.twiss_dense(
    hybrid_line,
    points_per_meter=10,
    betx=float(tw_cell.betx[0]), alfx=float(tw_cell.alfx[0]),
    bety=float(tw_cell.bety[0]), alfy=float(tw_cell.alfy[0]),
)

qfh.plot_hybrid_transition(
    tw_hybrid_from_strong_match,
    transition_s=hybrid_transition_s_m,
    title="Hybrid line: transition after 10 full periods, each of length 2L",
)

q8_reference_line = qfh.make_fodo_line(
    n_cells=1,
    k1=0.5,
    half_cell_length=L,
)
tw_q8_reference = qfh.twiss_periodic(q8_reference_line)
q8_boundary_evidence = qfh.cell_boundary_beta_beat_table(
    tw_hybrid_from_strong_match,
    tw_cell,
    tw_q8_reference,
    transition_s=hybrid_transition_s_m,
    cell_length=2 * L,
)
qfh.plot_cell_boundary_beta_beating(
    tw_hybrid_from_strong_match,
    tw_cell,
    tw_q8_reference,
    transition_s=hybrid_transition_s_m,
    cell_length=2 * L,
    title="Beta beating at QF centers separated by 2L",
)

q8_evidence_cell_indices = np.array([9, 10, 13, 15, 18, 20])
q8_evidence_s_m = q8_evidence_cell_indices * (2 * L)
display(
    q8_boundary_evidence[
        q8_boundary_evidence["s [m]"].isin(q8_evidence_s_m)
    ]
)

### Try it: change the downstream focusing strength

The first section remains fixed. Change the second-section strength and observe the mismatch at the transition.


In [ ]:
qfh.interactive_hybrid_transition(tw_cell)


### Worked example (ungraded): remove the strong-to-weak transition mismatch

The next two code cells are a worked example, not a numbered question. They demonstrate how to remove the beta beating diagnosed in Q8.

The upstream FODO cells use $|k_1|=0.60\ \mathrm{m}^{-2}$, while the downstream cells use $|k_1|=0.50\ \mathrm{m}^{-2}$. Each lattice has its own periodic Twiss solution. Joining them directly sends the upstream periodic solution into a lattice that expects different entrance values.

Here an $11\ \mathrm{m}$ section containing four adjustable quadrupoles is placed between the two lattices. The optimizer changes those four strengths until the endpoint values $(\beta_x,\alpha_x,\beta_y,\alpha_y)$ equal the periodic entrance values of the weaker downstream cell.

In the Xsuite matching call, the vary list names the four lattice variables the optimizer may change, while the targets list specifies four endpoint Twiss conditions. Xsuite iterates the strengths until all four residuals satisfy the requested tolerances.

The first output table checks those endpoint conditions. A difference near zero means that target has been reached. The plot compares the beta functions before and after optimization. The curves may differ substantially inside the section; this fully constrained problem requires the correct endpoint values, not a prescribed interior path.

In [ ]:
fodo_transition_k1_upstream = 0.6
fodo_transition_k1_downstream = 0.5

upstream_fodo_cell = qfh.make_fodo_line(n_cells=1, k1=fodo_transition_k1_upstream, half_cell_length=L)
downstream_fodo_cell = qfh.make_fodo_line(n_cells=1, k1=fodo_transition_k1_downstream, half_cell_length=L)

tw_upstream_fodo = qfh.twiss_periodic(upstream_fodo_cell)
tw_downstream_fodo = qfh.twiss_periodic(downstream_fodo_cell)

initial_strong_to_weak = qfh.initial_from_end(tw_upstream_fodo)
target_weak_fodo = qfh.initial_from_start(tw_downstream_fodo)

fodo_to_weak_match_section = qfh.make_match_section(initial_strength=fodo_transition_k1_upstream)
tw_strong_to_weak_before_dense = qfh.twiss_dense(
    fodo_to_weak_match_section,
    points_per_meter=80,
    **initial_strong_to_weak,
)

# Xsuite matching: specify the adjustable knobs and the desired endpoint optics.
with qfh.suppress_xsuite_output():
    fodo_to_weak_optimizer = fodo_to_weak_match_section.match(
        method="4d",
        vary=[
            xt.Vary("kQM1", step=1e-3, limits=(-3.0, 3.0)),
            xt.Vary("kQM2", step=1e-3, limits=(-3.0, 3.0)),
            xt.Vary("kQM3", step=1e-3, limits=(-3.0, 3.0)),
            xt.Vary("kQM4", step=1e-3, limits=(-3.0, 3.0)),
        ],
        targets=[
            xt.Target("betx", target_weak_fodo["betx"], at="_end_point", tol=1e-8),
            xt.Target("alfx", target_weak_fodo["alfx"], at="_end_point", tol=1e-8),
            xt.Target("bety", target_weak_fodo["bety"], at="_end_point", tol=1e-8),
            xt.Target("alfy", target_weak_fodo["alfy"], at="_end_point", tol=1e-8),
        ],
        n_steps_max=100,
        verbose=False,
        **initial_strong_to_weak,
    )

tw_strong_to_weak_after_dense = qfh.twiss_dense(
    fodo_to_weak_match_section,
    points_per_meter=80,
    **initial_strong_to_weak,
)

display(qfh.twiss_endpoint_comparison(tw_strong_to_weak_after_dense, target_weak_fodo))
qfh.plot_matching_comparison(
    tw_strong_to_weak_before_dense,
    tw_strong_to_weak_after_dense,
    "Match section from |k1| = 0.6 FODO to |k1| = 0.5 FODO",
)

# show final quad values
display(pd.DataFrame({
    "quadrupole": ["QM1", "QM2", "QM3", "QM4"],
    "final k1 [1/m^2]": [
        float(fodo_to_weak_match_section["kQM1"]),
        float(fodo_to_weak_match_section["kQM2"]),
        float(fodo_to_weak_match_section["kQM3"]),
        float(fodo_to_weak_match_section["kQM4"]),
    ],
}))


In [ ]:
upstream_fodo_section = qfh.make_fodo_line(n_cells=4, k1=fodo_transition_k1_upstream, half_cell_length=L)
downstream_fodo_section = qfh.make_fodo_line(n_cells=4, k1=fodo_transition_k1_downstream, half_cell_length=L)

fodo_match_weak_line, fodo_match_weak_sections = qfh.compose_labeled_sections([
    ("upstream FODO", "upstream", upstream_fodo_section),
    ("matching section", "match", fodo_to_weak_match_section),
    ("weaker FODO", "downstream", downstream_fodo_section),
])

tw_fodo_match_weak_dense = qfh.twiss_dense(
    fodo_match_weak_line,
    points_per_meter=50,
    **qfh.initial_from_start(tw_upstream_fodo),
)

qfh.plot_sectioned_twiss(
    tw_fodo_match_weak_dense,
    fodo_match_weak_sections,
    title="Twiss functions through FODO, matching section, and weaker FODO",
)


The combined plot places four upstream FODO cells, the optimized matching section, and four weaker FODO cells on one longitudinal axis; the shaded regions identify the three structures.

The important result is downstream of the matching section. The weaker cells begin with their own periodic Twiss values, so their beta pattern repeats immediately instead of developing the beating caused by an unmatched transition. Large or complicated beta variation inside the matching section is not itself a failure: the section's job is to deliver the required endpoint values.

Section F poses a different matching problem:

| Problem | Adjustable strengths | Independent endpoint conditions | What is prescribed? |
|---|---:|---:|---|
| Strong FODO $\rightarrow$ weak FODO | 4 | 4 | All four downstream Twiss values |
| Section F round-waist target | 4 | 3 | $\beta_x=\beta_y$ and $\alpha_x=\alpha_y=0$; the common beta is free |

Keep that distinction in mind when deciding whether the next optimizer result should be unique.

### F) Matching-section insert: a round-waist target

A real accelerator contains insertions, injection and extraction regions, experiments, and other sections that are not repeated FODO cells. A **matching section** uses adjustable quadrupoles to transform incoming Twiss parameters into desired outgoing Twiss parameters.

The incoming beam is taken from the end of the matched FODO cell. The target at the end of the matching section is

$$
\beta_x=\beta_y, \qquad \alpha_x=0, \qquad \alpha_y=0.
$$

The transverse covariance matrix in either uncoupled plane is

$$
\Sigma=\epsilon
\begin{pmatrix}
\beta & -\alpha\\
-\alpha & \gamma
\end{pmatrix},
\qquad
\gamma=\frac{1+\alpha^2}{\beta},
\qquad
\det\Sigma=\epsilon^2.
$$

The section geometry is

`1 m drift — QM1 — 2 m drift — QM2 — 2 m drift — QM3 — 2 m drift — QM4 — 2 m drift`,

with four $0.50\ \mathrm{m}$ quadrupoles. The layout table is displayed before the manual sliders.

**Q9) Count the independently adjustable quadrupole strengths and the independent endpoint conditions. Is this matching problem underconstrained, exactly constrained, or overconstrained? Should the optimized strengths be unique? Explain what role the unconstrained common final beta value and the optimizer's starting point can play.**

### Try it: match by hand

Inspect the element layout, then *attempt* to adjust the four quadrupole strengths manually. Try to reduce all three endpoint residuals: $\beta_x-\beta_y$, $\alpha_x$, and $\alpha_y$. Since we have four knobs and only three conditions to solve for, there are a family of solutions rather than one single unique solution. If you want a starting point, try setting the first two knobs to [0.21, -0.72] and then find values for the last two knobs that can get close to the goal.

Record at least one knob change that improves one residual but worsens another. This is the tradeoff the optimizer must navigate. **You are not expected to actually be able to satisfy all three conditions by hand.** The goal is to just play around with the knobs and understand the complex response of the coupled optics knobs before using the optimizer.

Note that these sliders can be set to both positive and negative to make focusing and defocusing quadruopoles.

In [ ]:
initial_from_fodo_end = qfh.fodo_end_twiss_dict(tw_cell)

manual_match_layout = qfh.make_match_section()
display(
    manual_match_layout.get_table().to_pandas()[
        ["name", "s_start", "s_end", "element_type"]
    ]
)

qfh.interactive_manual_match(initial_from_fodo_end)

### Optimized matching solution

The next cell constructs the four-quadrupole section, runs the Xsuite optimizer, and stores the Twiss results and summary tables used below. Because Q9 identifies a free degree of freedom, this is one valid solution selected from a family by the initial strengths and the optimizer trajectory; it is not the only possible match.

The optimizer is run once. Each numbered question is then placed immediately beside the display it asks you to interpret.

In [ ]:
# Build the matching section and solve the three endpoint constraints once.
initial_from_fodo_end = qfh.fodo_end_twiss_dict(tw_cell)

match_env = xt.Environment()
match_env["k1_fodo"] = env["k1_fodo"]
match_env.new("DM0", xt.Drift, length=1.0)

match_components = ["DM0"]
for i, sign in enumerate([+1, -1, +1, -1], start=1):
    match_env[f"kQM{i}"] = sign * match_env["k1_fodo"]
    match_env.new(f"QM{i}", xt.Quadrupole, length=0.5, k1=f"kQM{i}")
    match_env.new(f"DM{i}", xt.Drift, length=2.0)
    match_components.extend([f"QM{i}", f"DM{i}"])

match_section = match_env.new_line(name="match_section", components=match_components)
match_section.particle_ref = xt.Particles(p0c=1e9, mass0=xt.ELECTRON_MASS_EV, q0=-1)
match_section.build_tracker()

tw_match_before = match_section.twiss(method="4d", **initial_from_fodo_end)
tw_match_before_dense = qfh.twiss_dense(match_section, points_per_meter=80, **initial_from_fodo_end)

with qfh.suppress_xsuite_output():
    opt = match_section.match(
        method="4d",
        vary=[
            xt.Vary("kQM1", step=1e-3, limits=(-3.0, 3.0)),
            xt.Vary("kQM2", step=1e-3, limits=(-3.0, 3.0)),
            xt.Vary("kQM3", step=1e-3, limits=(-3.0, 3.0)),
            xt.Vary("kQM4", step=1e-3, limits=(-3.0, 3.0)),
        ],
        targets=[
            xt.Target("alfx", 0.0, at="_end_point", tol=1e-8),
            xt.Target("alfy", 0.0, at="_end_point", tol=1e-8),
            xt.Target(
                lambda tw: tw["betx", "_end_point"] - tw["bety", "_end_point"],
                0.0,
                tol=1e-8,
                tag="betx_minus_bety",
            ),
        ],
        n_steps_max=80,
        verbose=False,
        **initial_from_fodo_end,
    )

tw_match_after = match_section.twiss(method="4d", **initial_from_fodo_end)
tw_match_after_dense = qfh.twiss_dense(match_section, points_per_meter=80, **initial_from_fodo_end)

endpoint_measurements = pd.DataFrame({
    "Twiss quantity": ["beta_x [m]", "alpha_x", "beta_y [m]", "alpha_y"],
    "entrance": [
        initial_from_fodo_end["betx"], initial_from_fodo_end["alfx"],
        initial_from_fodo_end["bety"], initial_from_fodo_end["alfy"],
    ],
    "unoptimized exit": [
        tw_match_before.betx[-1], tw_match_before.alfx[-1],
        tw_match_before.bety[-1], tw_match_before.alfy[-1],
    ],
    "optimized exit": [
        tw_match_after.betx[-1], tw_match_after.alfx[-1],
        tw_match_after.bety[-1], tw_match_after.alfy[-1],
    ],
})
optimized_strengths = pd.DataFrame({
    "quadrupole": [f"QM{i}" for i in range(1, 5)],
    "k1 [1/m^2]": [float(match_section.vars[f"kQM{i}"]._value) for i in range(1, 5)],
})

display(endpoint_measurements)
display(optimized_strengths)

Shown above are the "correct answers" for the quadrupole strengths. Try putting them in to the sliders above and see if it really fulfills the requirements.

### **Q10) Tracked horizontal phase space and beam-size change**

The plot directly below follows the same particles from the entrance, through the four matching quadrupoles, to the exit. All panels use the same axis scales, and the black curve is the local one-RMS ellipse.

Compare the entrance and exit position widths, angle widths, and orientations, then identify what happens to the tilt/correlation at intermediate locations. Use

$$
\Sigma_{xx}=\epsilon\beta_x,
\qquad
\Sigma_{xx'}=-\epsilon\alpha_x,
\qquad
\Sigma_{x'x'}=\epsilon\gamma_x
$$

to explain those changes. Ideal uncoupled linear transport preserves emittance. How can the section reshape the ellipse without cooling or heating the beam?

Finally, use the entrance and optimized-exit beta values to calculate

$$
\frac{\sigma_{u,\mathrm{out}}}{\sigma_{u,\mathrm{in}}}
=\sqrt{\frac{\beta_{u,\mathrm{out}}}{\beta_{u,\mathrm{in}}}},
\qquad u=x,y.
$$

Which plane becomes narrower at the exit, and which becomes wider?

In [ ]:
tracked_distribution = qfh.track_distribution_along_line(
    match_section,
    initial_from_fodo_end,
    n_particles=2500,
    seed=11,
)
qfh.plot_phase_space_filmstrip(
    tracked_distribution,
    tw_match_after,
    title="Tracked horizontal phase space through the matching section",
)

qfh.plot_linked_matching_section_phase_space(
    tracked_distribution,
    tw_match_after_dense,
    match_section,
    title="Linked beam envelope and horizontal phase space",
    sample_step_m=0.1,
)

display(endpoint_measurements[["Twiss quantity", "entrance", "optimized exit"]])

### Observation checkpoint (ungraded): before and after optimization

Use the beta-function plot directly below to compare the unoptimized and optimized curves through the interior and at the exit.

What did the optimizer visibly change? Does the optimized solution approach $\beta_x=\beta_y$ at the endpoint? Explain why a successful endpoint match does **not** require the optimized and unoptimized curves to coincide inside the matching section.

In [ ]:
qfh.plot_matching_comparison(tw_match_before_dense, tw_match_after_dense)

### **Q11) Quantitative endpoint verification**

We can define $m_\beta$ as a metric to measure the asymmetry in $\beta$. Use the `endpoint_measurements` table directly below to calculate

$$
m_\beta=\frac{|\beta_x-\beta_y|}{(\beta_x+\beta_y)/2}
$$

for the unoptimized and optimized exits. By approximately how many orders of magnitude was it reduced? What two additional table entries must be checked before declaring all three endpoint constraints satisfied?

Explain why the final digits of a near-zero residual represent solver tolerance rather than experimental precision.

In [ ]:
display(endpoint_measurements)

### **Q12) Optimized quadrupole strengths**

Use the `optimized_strengths` table directly below. Before doing the arithmetic, use $|k_1|$ to predict which quadrupole should have the shortest focal length. Then, with $L_q=0.50\ \mathrm{m}$, calculate

$$
|f_i|=\frac{1}{|k_{1,i}|L_q}
$$

for QM1–QM4 and rank the lenses from strongest to weakest. What does the sign of $k_1$ tell you that its magnitude does not? Why does this matching section use unequal strengths instead of the repeated equal-magnitude pattern of a periodic FODO cell?

In [ ]:
display(optimized_strengths)

### G) Advanced extension: global versus local periodicity

Return to the 100 m hybrid line from Section E. We now **mathematically imagine repeating the entire line as one superperiod**:

$$
P=20(2L)=40L=100\ \mathrm{m}.
$$

Asking Xsuite for this full-period periodic solution does not make the strong and weak cells—each of full length $2L=5.00\ \mathrm{m}$—locally identical, and it does not replace the transition match demonstrated above.

A periodic solution over length $P$ satisfies the same Twiss parameters at $s$ and $s+P$.

**Q13) Use the raw start/end Twiss values as evidence that this solution is matched for the full superperiod $P=40L=100\ \mathrm{m}$. Then explain why closure after $P$ does not imply closure after one full cell $2L$, and why no single Twiss solution can be $2L$-periodic throughout this two-strength lattice.**

In [ ]:
hybrid_superperiod_m = 20 * (2 * L)
assert np.isclose(hybrid_line.get_length(), hybrid_superperiod_m)

tw_hybrid_periodic = qfh.twiss_periodic(hybrid_line)
tw_hybrid_periodic_dense = qfh.twiss_dense(
    hybrid_line,
    points_per_meter=10,
    betx=float(tw_hybrid_periodic.betx[0]),
    alfx=float(tw_hybrid_periodic.alfx[0]),
    bety=float(tw_hybrid_periodic.bety[0]),
    alfy=float(tw_hybrid_periodic.alfy[0]),
)

qfh.plot_hybrid_transition(
    tw_hybrid_periodic_dense,
    transition_s=hybrid_transition_s_m,
    title=f"Periodic solution of full superperiod P = {hybrid_superperiod_m:g} m = 40L",
)

pd.DataFrame({
    "Twiss quantity": ["beta_x [m]", "alpha_x", "beta_y [m]", "alpha_y"],
    "s = 0": [
        tw_hybrid_periodic.betx[0], tw_hybrid_periodic.alfx[0],
        tw_hybrid_periodic.bety[0], tw_hybrid_periodic.alfy[0],
    ],
    f"s = P = {hybrid_superperiod_m:g} m": [
        tw_hybrid_periodic.betx[-1], tw_hybrid_periodic.alfx[-1],
        tw_hybrid_periodic.bety[-1], tw_hybrid_periodic.alfy[-1],
    ],
})

### H) Optional design project

**Q14) Design a purposeful injection insertion.**

The supplied baseline uses a matching section followed by a mirrored copy and contains a $4.0\ \mathrm{m}$ field-free central drift formed by the two adjacent DM4 elements. It is only a symmetry check, not a completed injection design.

Give the central region a concrete optical purpose:

1. Increase the total field-free central drift from $4.0\ \mathrm{m}$ to $6.0\ \mathrm{m}$ and place an injection marker at its center.
2. Give the outbound section independent knobs and match a round waist at that marker: $\beta_x=\beta_y$ and $\alpha_x=\alpha_y=0$. The common beta remains free.
3. Give the return section four **independent** quadrupole strengths and match the exit to the downstream periodic FODO values $(\beta_x,\alpha_x,\beta_y,\alpha_y)$.

Submit:

- a lattice sketch or element table identifying the 6 m drift and injection marker;
- the full beta-function plot with the marker indicated;
- the center residuals $\beta_x-\beta_y$, $\alpha_x$, and $\alpha_y$;
- the four exit residuals relative to the downstream periodic FODO target;
- both sets of optimized quadrupole strengths;
- a short assessment of whether the design satisfies the center and exit requirements and whether the peak beta values are reasonable.

Simply rerunning the unchanged symmetric baseline is not a completed design.

In [ ]:
baseline_insertion = qfh.make_injection_insertion_line(fodo_cell, match_section)
tw_baseline_insertion = qfh.twiss_dense(
    baseline_insertion,
    points_per_meter=60,
    **initial_from_fodo_end,
)
qfh.plot_twiss(
    tw_baseline_insertion,
    "Symmetric insertion baseline for Q14 (not a completed injection design)",
    show_lattice=False,
)

pd.DataFrame({
    "Twiss quantity": ["beta_x [m]", "alpha_x", "beta_y [m]", "alpha_y"],
    "entrance": [
        tw_baseline_insertion.betx[0], tw_baseline_insertion.alfx[0],
        tw_baseline_insertion.bety[0], tw_baseline_insertion.alfy[0],
    ],
    "exit": [
        tw_baseline_insertion.betx[-1], tw_baseline_insertion.alfx[-1],
        tw_baseline_insertion.bety[-1], tw_baseline_insertion.alfy[-1],
    ],
})
